In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

# Load data
df = pd.read_csv("processed_hdfs.csv")

X = df.drop(["BlockId","Label"], axis=1)
y = df["Label"]

# Feature filtering
selector = VarianceThreshold(threshold=0.01)
X = selector.fit_transform(X)

# Log transform
X = np.log1p(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -------- Train only on NORMAL logs --------
X_train_normal = X_train[y_train == 0]

# Autoencoder architecture
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(32, activation="relu")(input_layer)
encoded = Dense(16, activation="relu")(encoded)

decoded = Dense(32, activation="relu")(encoded)
decoded = Dense(input_dim, activation="sigmoid")(decoded)

autoencoder = Model(input_layer, decoded)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

# Train model
history = autoencoder.fit(
    X_train_normal,
    X_train_normal,
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_split=0.1
)

# Reconstruction
X_test_pred = autoencoder.predict(X_test)

# Reconstruction error
mse = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

# Threshold
threshold = np.percentile(mse, 95)

y_pred = (mse > threshold).astype(int)

# Evaluation
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ROC AUC
auc = roc_auc_score(y_test, mse)
print("ROC AUC:", auc)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, mse)

plt.plot(fpr, tpr, label=f'Autoencoder (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'tensorflow'

In [2]:
import sys
print(sys.executable)

/opt/anaconda3/envs/dl_env/bin/python
